# Prompt Engineering with LLM APIs using a chainable Template

### Author: Alex Pang (alex.pang@temple.edu)

#### Using Prompt Templates with Google Gemini

In this notebook, we'll continue from the previous Prompt Engineering notebook

In [3]:
# Install the Google Generative AI library
#!pip install -q google-genai

In [4]:
from google import genai
import os
from typing import Dict, List

# select your model
model = "gemini-2.5-flash"
client = genai.Client(api_key=os.getenv('GOOGLE_API_KEY'))

In [5]:
# Test setup
response = client.models.generate_content(
    model= model,
    contents="Hello!"
)
print(response.text)
print("✓ Setup complete!")

Hello! How can I help you today?
✓ Setup complete!


## Chainable Prompt Template

In the Prompt Engineering notebook, I show you a basic Prompt Template class that you can reuse to capture the few basic prompt engineering
techniques. The design was pretty basic; you supply all the necessary information in the constructors.

Here, we import a chainable design. You can examine the python script 

In [6]:
from prompt_template import PromptTemplate

#### You should import PromptTemplate from the prompt_template script, but I am going to paste it here just so you can see everything

In [14]:
from typing import List, Dict

from typing import List, Dict

class PromptTemplate:
    """
    A chainable prompt template class.
    
    Each method returns a new PromptTemplate instance, allowing fluent
    method chaining to build complex prompts incrementally.
    """
    
    def __init__(self, template: str, **kwargs):
        """
        Initialize the template.
        
        Args:
            template: The base prompt text (can include {placeholders})
            **kwargs: Default values for placeholders
        """
        self.template = template
        self.default_params = kwargs
    
    
    def with_role(self, role: str) -> 'PromptTemplate':
        """
        Add a role/persona to the prompt.
        
        Args:
            role: Description of the role (e.g., "a physics professor")
            
        Returns:
            A new PromptTemplate with the role prepended
        """
        new_template = f"You are {role}.\n\n{self.template}"
        return PromptTemplate(new_template, **self.default_params)
    
    def with_output_format(self, format_spec: str) -> 'PromptTemplate':
        """
        Specify the desired output format.
        
        Args:
            format_spec: Description of expected output format
            
        Returns:
            A new PromptTemplate with format instructions appended
        """
        new_template = f"{self.template}\n\nProvide your response in the following format:\n{format_spec}"
        return PromptTemplate(new_template, **self.default_params)
    
    def with_context(self, context: str) -> 'PromptTemplate':
        """
        Add contextual information to the prompt.
        
        Args:
            context: Background information or context
            
        Returns:
            A new PromptTemplate with context added
        """
        new_template = f"Context: {context}\n\n{self.template}"
        return PromptTemplate(new_template, **self.default_params)
    
    def with_constraints(self, constraints: List[str]) -> 'PromptTemplate':
        """
        Add constraints to the prompt.
        
        Args:
            constraints: List of constraints to follow
            
        Returns:
            A new PromptTemplate with constraints appended
        """
        constraints_text = "\n".join([f"- {c}" for c in constraints])
        new_template = f"{self.template}\n\nConstraints:\n{constraints_text}"
        return PromptTemplate(new_template, **self.default_params)
    
    def with_examples(self, examples: List[Dict[str, str]]) -> 'PromptTemplate':
        """
        Add few-shot examples to the prompt.
        
        Args:
            examples: List of dicts with 'input' and 'output' keys
            
        Returns:
            A new PromptTemplate with examples inserted
        """
        examples_text = "\n\n".join([
            f"Example {i+1}:\nInput: {ex['input']}\nOutput: {ex['output']}"
            for i, ex in enumerate(examples)
        ])
        new_template = f"Examples:\n{examples_text}\n\n{self.template}"
        return PromptTemplate(new_template, **self.default_params)
    
    def with_chain_of_thought(self) -> 'PromptTemplate':
        """
        Add chain-of-thought instruction.
        
        Returns:
            A new PromptTemplate with step-by-step reasoning instruction
        """
        new_template = f"{self.template}\n\nThink through this step-by-step and show your reasoning."
        return PromptTemplate(new_template, **self.default_params)
    

    def full_prompt(self, **kwargs) -> str:
        """
        Format the template with provided parameters.
        
        Args:
            **kwargs: Values to fill in placeholders
            
        Returns:
            The formatted prompt string
        """
        params = {**self.default_params, **kwargs}
        return self.template.format(**params)
    

    def generate(self, client, model, **kwargs) -> str:
        """
        Use Google genai client to generate response from an input prompt
        """
        full_prompt = self.full_prompt(**kwargs)
        response = client.models.generate_content(model = model, contents = full_prompt)
        return response.text
    
    def __str__(self) -> str:
        """Return the current template string."""
        return self.template
    
    def __repr__(self) -> str:
        """Return a detailed representation."""
        return f"PromptTemplate(template={self.template[:50]}..., params={self.default_params})"


#### Example 1: Basic Task Only

In [8]:
print("\n" + "=" * 70)
print("EXAMPLE 1: Basic Task Only")
print("=" * 70)

# Create a simple template
prompt = PromptTemplate("Explain Quantum Computing")

print("\n[Constructed Prompt]")
print(prompt.full_prompt())


EXAMPLE 1: Basic Task Only

[Constructed Prompt]
Explain Quantum Computing


In [9]:
print("\n[Response]")
response = prompt.generate(client, model)
print(response)


[Response]
Quantum computing is a fundamentally new type of computing that leverages the principles of quantum mechanics to solve certain complex problems that are intractable for classical computers.

Let's break it down:

## 1. Classical Computers vs. Quantum Computers

*   **Classical Computers:** Store information as **bits**, which can be either a **0 or a 1**. They process information sequentially, following a defined set of logic gates.
*   **Quantum Computers:** Store information as **qubits**, which are the quantum equivalent of bits. Qubits have some mind-bending properties:

## 2. The Core Principles of Quantum Computing

Quantum computers gain their power from three main quantum mechanical phenomena:

1.  **Superposition:**
    *   **Classical Bit:** Is either 0 or 1.
    *   **Qubit:** Can be 0, 1, or **both 0 and 1 simultaneously** (in a probabilistic sense). Imagine a spinning coin that is neither heads nor tails until it lands. A qubit in superposition is like that spi

#### Example 2: Adding a Role

In [10]:
print("\n" + "=" * 70)
print("EXAMPLE 2: Adding a Role")
print("=" * 70)

prompt = (PromptTemplate("Explain Quantum Computing")
    .with_role("a physics professor who makes complex topics simple"))

print("\n[Constructed Prompt]")
print(prompt.full_prompt())


EXAMPLE 2: Adding a Role

[Constructed Prompt]
You are a physics professor who makes complex topics simple.

Explain Quantum Computing


#### You can chain any choice you want to turn on

In [11]:
print("\n" + "=" * 70)
print("EXAMPLE 5: With Constraints")
print("=" * 70)

prompt = (PromptTemplate("Explain Quantum Computing")
    .with_role("a science communicator")
    .with_constraints([
        "Keep the explanation under 150 words",
        "Use at least one analogy from everyday life",
        "Avoid mathematical equations",
        "End with a thought-provoking question"
    ]))

print("\n[Constructed Prompt]")
print(prompt.full_prompt())


EXAMPLE 5: With Constraints

[Constructed Prompt]
You are a science communicator.

Explain Quantum Computing

Constraints:
- Keep the explanation under 150 words
- Use at least one analogy from everyday life
- Avoid mathematical equations
- End with a thought-provoking question


In [12]:
print("\n" + "=" * 70)
print("EXAMPLE 6: With Few-Shot Examples")
print("=" * 70)

prompt = (PromptTemplate("Explain Quantum Computing")
    .with_examples([
        {
            "input": "Explain Machine Learning",
            "output": "Imagine teaching a dog tricks by giving treats for correct behavior. Machine Learning works similarly—we 'reward' computer programs when they make correct predictions, helping them learn patterns from data without explicit programming."
        },
        {
            "input": "Explain Blockchain",
            "output": "Think of a shared Google Doc that everyone can view but no one can secretly edit. Each change is recorded permanently with a timestamp. Blockchain is this concept applied to financial transactions and data—transparent, permanent, and trustworthy without needing a central authority."
        }
    ]))

print("\n[Constructed Prompt]")
print(prompt.full_prompt())


EXAMPLE 6: With Few-Shot Examples

[Constructed Prompt]
Examples:
Example 1:
Input: Explain Machine Learning
Output: Imagine teaching a dog tricks by giving treats for correct behavior. Machine Learning works similarly—we 'reward' computer programs when they make correct predictions, helping them learn patterns from data without explicit programming.

Example 2:
Input: Explain Blockchain
Output: Think of a shared Google Doc that everyone can view but no one can secretly edit. Each change is recorded permanently with a timestamp. Blockchain is this concept applied to financial transactions and data—transparent, permanent, and trustworthy without needing a central authority.

Explain Quantum Computing


#### Chain everything

In [13]:
print("\n" + "=" * 70)
print("EXAMPLE 8: Full Method Chain - All Techniques")
print("=" * 70)

prompt = (PromptTemplate("Explain Quantum Computing")
    .with_role("an award-winning science journalist")
    .with_context("Writing for a tech newsletter read by busy professionals")
    .with_examples([
        {
            "input": "Explain 5G",
            "output": "Your current phone internet is like a two-lane road. 5G is a 100-lane superhighway. This isn't just about faster cat videos—it's the infrastructure for self-driving cars, remote surgery, and smart cities. The one thing to remember: 5G is less about speed and more about enabling technologies that don't exist yet."
        }
    ])
    .with_output_format("""- Hook (1 attention-grabbing sentence)
- Core Idea (2-3 sentences, use an analogy)  
- Why It Matters (1-2 sentences on real-world impact)
- The Takeaway (1 memorable closing sentence)""")
    .with_constraints([
        "Total response: 100-150 words",
        "Conversational, not academic tone",
        "No jargon without immediate explanation"
    ]))

print("\n[Constructed Prompt]")
print(prompt.full_prompt())


EXAMPLE 8: Full Method Chain - All Techniques

[Constructed Prompt]
Examples:
Example 1:
Input: Explain 5G
Output: Your current phone internet is like a two-lane road. 5G is a 100-lane superhighway. This isn't just about faster cat videos—it's the infrastructure for self-driving cars, remote surgery, and smart cities. The one thing to remember: 5G is less about speed and more about enabling technologies that don't exist yet.

Context: Writing for a tech newsletter read by busy professionals

You are an award-winning science journalist.

Explain Quantum Computing

Provide your response in the following format:
- Hook (1 attention-grabbing sentence)
- Core Idea (2-3 sentences, use an analogy)  
- Why It Matters (1-2 sentences on real-world impact)
- The Takeaway (1 memorable closing sentence)

Constraints:
- Total response: 100-150 words
- Conversational, not academic tone
- No jargon without immediate explanation
